In [1]:
from pathlib import Path

In [2]:
BASE = Path("/Users/joregan/Playing/work-2026/arctic")

In [6]:
RXR = BASE / "cmu_us_rxr_arctic" / "etc" / "txt.done.data"

In [4]:
def read_text(file: Path):
    text_pairs = {}
    with open(file) as f:
        for line in f.readlines():
            text_start = line.find('"')
            text_end = line.rfind('"')
            text = line[text_start + 1:text_end]
            text_id = line[1:text_start].strip()
            text_pairs[text_id] = text
    return text_pairs

In [10]:
rxr_ids = {k: v for k, v in read_text(RXR).items()}

In [27]:
def norm_word(word: str):
    if word in ["--", "-"]:
        return ""
    return word.lower().strip(".,!?;:\"()")

def norm_sentence(sentence: str):
    return " ".join([norm_word(w) for w in sentence.split(" ") if norm_word(w) != ""])

In [28]:
reverse_rxr_ids = {norm_sentence(v): k for k, v in rxr_ids.items()}

In [34]:
remapped = {}

for subset in BASE.glob("cmu_us_*_arctic"):
    person = subset.name.split("_")[2]
    for file in subset.glob("etc/txt.done.data"):
        for pair in read_text(file).items():
            sentence = norm_sentence(pair[1])
            if sentence not in reverse_rxr_ids:
                if pair[0] not in rxr_ids:
                    rxr_ids[pair[0]] = pair[1]
                    reverse_rxr_ids[sentence] = pair[0]
                else:
                    print(f"Missing: {person} {sentence}")
            else:
                if sentence in reverse_rxr_ids and reverse_rxr_ids[sentence] != pair[0]:
                    if not person in remapped:
                        remapped[person] = {}
                    remapped[person][pair[0]] = reverse_rxr_ids[sentence]
                elif person == "awb" and "-" in sentence:
                    sentence_no_hyphen = sentence.replace("-", "")
                    if sentence_no_hyphen in reverse_rxr_ids:
                        if sentence_no_hyphen in reverse_rxr_ids and reverse_rxr_ids[sentence_no_hyphen] != pair[0]:
                            if not person in remapped:
                                remapped[person] = {}
                            remapped[person][pair[0]] = reverse_rxr_ids[sentence_no_hyphen]
            

Missing: jmk lord fitzhugh is the key to this whole situation
Missing: awb now he held the whip-hand brokaw had acknowledged his own surrender
Missing: awb the night-glow was treacherous to shoot by
Missing: awb he went down in mid-stream searching the shadows of both shores
Missing: awb you you would not--keep the truth from me
Missing: awb good-by pierre he shouted
Missing: awb they are to attack your camp to- morrow night
Missing: awb to-morrow i'm going after that bear he said
Missing: awb why the average review is more nauseating than cod-liver oil
Missing: awb they ought to pass here some time to-day
Missing: awb o'brien had been a clean-living young man with ideals
Missing: awb soaked in sea-water they offset the heat-rays
Missing: awb there are no kiddies and half-grown youths among them
Missing: awb here in the mid-morning the first casualty occurred
Missing: awb i tell you i am disgusted with this adventure tom-foolery and rot
Missing: awb this is no place fer you
Missing: aw

In [36]:
remapped

{'slt': {'arctic_a0456': 'arctic_c0035'},
 'ljm': {'arctic_a0456': 'arctic_c0035'},
 'eey': {'arctic_a0456': 'arctic_c0035'},
 'ahw': {'arctic_a0456': 'arctic_c0035'},
 'gka': {'arctic_a0456': 'arctic_c0035'},
 'lnh': {'arctic_a0456': 'arctic_c0035'},
 'axb': {'arctic_a0456': 'arctic_c0035'},
 'rxr': {'arctic_a0456': 'arctic_c0035'},
 'slp': {'arctic_a0456': 'arctic_c0035'},
 'ksp': {'arctic_a0456': 'arctic_c0035'},
 'fem': {'arctic_a0456': 'arctic_c0035'},
 'aup': {'arctic_a0456': 'arctic_c0035'},
 'clb': {'arctic_a0456': 'arctic_c0035'},
 'bdl': {'arctic_a0456': 'arctic_c0035'},
 'awb': {'arctic_a0075': 'arctic_a0074',
  'arctic_a0076': 'arctic_a0075',
  'arctic_a0077': 'arctic_a0076',
  'arctic_a0078': 'arctic_a0077',
  'arctic_a0079': 'arctic_c0026',
  'arctic_a0080': 'arctic_a0078',
  'arctic_a0081': 'arctic_a0079',
  'arctic_a0082': 'arctic_a0080',
  'arctic_a0083': 'arctic_a0081',
  'arctic_a0084': 'arctic_a0082',
  'arctic_a0085': 'arctic_a0083',
  'arctic_a0086': 'arctic_a0084

In [35]:
import json
with open("arctic-remapped-ids.json", "w") as f:
    json.dump(remapped, f, indent=4)